# Fatigue-Aware Tourist Route Planning using A* Search
This notebook implements an A* informed search over a real OpenStreetMap road network to find the optimal tourist walking/cycling route while managing human stamina and automatically inserting rest stops at famous tourist landmarks.

In [ ]:
!pip install osmnx pandas folium networkx -q


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: C:\Users\adity\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import heapq
import pandas as pd
import osmnx as ox
import folium
from math import radians, sin, cos, sqrt, atan2

print("Downloading road network for Pune...")
G = ox.graph_from_point((18.5308, 73.8475), dist=15000, network_type="drive")
G = ox.convert.to_undirected(G)
print("Nodes:", len(G.nodes), "Edges:", G.number_of_edges())

Nodes: 63235 Edges: 78618


In [ ]:
stations_df = pd.read_csv("pune-tourist-landmarks.csv")
city_landmarks = stations_df[stations_df['city'].str.contains("Pune", case=False, na=False)].copy()
city_landmarks['latitude'] = pd.to_numeric(city_landmarks['latitude'], errors='coerce')
city_landmarks['longitude'] = pd.to_numeric(city_landmarks['longitude'], errors='coerce')
city_landmarks = city_landmarks.dropna(subset=['latitude', 'longitude'])

landmark_nodes = [
    ox.distance.nearest_nodes(G, row['longitude'], row['latitude'])
    for _, row in city_landmarks.iterrows()
]
landmark_set = set(landmark_nodes)
print(f"Loaded {len(landmark_set)} tourist landmarks.")

Loaded 15 tourist landmarks.


In [ ]:
def haversine(n1, n2):
    lat1, lon1 = G.nodes[n1]['y'], G.nodes[n1]['x']
    lat2, lon2 = G.nodes[n2]['y'], G.nodes[n2]['x']
    R = 6371
    dlat, dlon = radians(lat2-lat1), radians(lon2-lon1)
    h = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(h), sqrt(1-h))

def heuristic(node, goal):
    return haversine(node, goal)

In [ ]:
STAMINA_LIMIT_KM = 25

def a_star_tourist(start, goal):
    pq = [(0, start, STAMINA_LIMIT_KM, [start], 0, 0)]
    visited = {}
    expanded = 0
    
    while pq:
        priority, node, stamina, path, dist, stops = heapq.heappop(pq)
        expanded += 1
        
        if node == goal:
            print(f"Scenic route found: {len(path)} nodes | Distance: {dist:.2f} km")
            print(f"Rest stops taken: {stops} | Nodes expanded: {expanded}")
            return path
            
        key = (node, round(stamina))
        if key in visited and visited[key] <= dist:
            continue
        visited[key] = dist
        
        for neighbor in G.neighbors(node):
            edge_data = G.get_edge_data(node, neighbor)
            length_km = min(d.get('length', 0) for d in edge_data.values()) / 1000
            
            new_stamina = stamina - length_km
            new_stops = stops
            
            # Always take a break and sightsee if we pass a landmark!
            if node in landmark_set:
                new_stamina = STAMINA_LIMIT_KM
                new_stops += 1
                
            if new_stamina < 0:
                continue
                
            new_dist = dist + length_km
            new_priority = new_dist + heuristic(neighbor, goal)
            
            heapq.heappush(pq, (new_priority, neighbor, new_stamina, path+[neighbor], new_dist, new_stops))
            
    print("No feasible route within stamina constraints.")
    return None

In [ ]:
start_node = ox.distance.nearest_nodes(G, 73.8475, 18.5308)
goal_node  = ox.distance.nearest_nodes(G, 73.7380, 18.5910)

path = a_star_tourist(start_node, goal_node)

if path:
    total_km = sum(min(d.get('length', 0) for d in G.get_edge_data(path[i], path[i+1]).values()) / 1000 for i in range(len(path)-1))
    landmarks_on_route = [n for n in path if n in landmark_set]
    straight_line_km = haversine(start_node, goal_node)
    detour_pct = ((total_km - straight_line_km) / straight_line_km) * 100
    est_time_min = (total_km / 15) * 60
    rest_time_min = len(landmarks_on_route) * 30

    print("\n=== YOUR TOURIST TRIP SUMMARY ===")
    print("─" * 40)
    print(f"Distance to travel   : {total_km:.1f} km")
    print(f"Estimated travel time: {int(est_time_min)} min")
    if landmarks_on_route:
        print(f"Rest stops needed    : {len(landmarks_on_route)}")
        print(f"Extra time for sightseeing: ~{rest_time_min:.0f} min")
        print(f"Total trip time (incl. stops): ~{int(est_time_min + rest_time_min)} min")
    else:
        print("Rest stops needed: 0 — your stamina is enough for this trip!")
    print(f"\nThis route is {detour_pct:.0f}% longer than a straight-line path.\n")

    # Generate visual map exactly like the reference
    route_coords = [(G.nodes[n]['y'], G.nodes[n]['x']) for n in path]
    m = folium.Map(location=route_coords[0], zoom_start=14)
    
    # Plot route
    folium.PolyLine(route_coords, color="purple", weight=5).add_to(m)
    
    # Start and Goal markers
    folium.Marker(route_coords[0], icon=folium.Icon(color="red"), popup="Start").add_to(m)
    folium.Marker(route_coords[-1], icon=folium.Icon(color="blue"), popup="Goal").add_to(m)
    
    # Landmark markers
    for _, row in city_landmarks.iterrows():
        folium.Marker([row['latitude'], row['longitude']], icon=folium.Icon(color="orange", icon="camera", prefix='fa')).add_to(m)

    # Save and display cleanly
    m.save("tourist_route_map.html")
    display(m)


Scenic route found: 95 nodes | Distance: 14.69 km
Rest stops taken: 1 | Nodes expanded: 16605

=== YOUR TOURIST TRIP SUMMARY ===
────────────────────────────────────────
Distance to travel   : 14.7 km
Estimated travel time: 58 min
Rest stops needed    : 1
Extra time for sightseeing: ~30 min
Total trip time (incl. stops): ~88 min

This route is 10% longer than a straight-line path.

